# Three Content-Based Recommenders
Compare cosine similarity, Word2Vec, and BERT embeddings on the same book dataset.

In [ ]:
import numpy as np
import tqdm as notebook_tqdm

from recommender import (
    BertEmbeddingRecommender,
    CosineSimilarityRecommender,
    PrecisionRecallEvaluator,
    Word2VecRecommender,
    choose_example_titles,
    load_books,
    print_examples,
 )

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BOOKS_DF = load_books()

## Cosine Similarity
This model uses TF-IDF vectors and cosine similarity as the baseline content-based recommender.

In [ ]:
cosine_recommender = CosineSimilarityRecommender(BOOKS_DF)
cosine_example_titles = choose_example_titles(BOOKS_DF, count=2)
print_examples("Cosine Similarity Examples", cosine_recommender, cosine_example_titles, num_recommendations=5)

## Word2Vec
This model averages learned Word2Vec embeddings for each book and ranks books by embedding similarity.

In [ ]:
word2vec_recommender = Word2VecRecommender(BOOKS_DF)
word2vec_example_titles = choose_example_titles(BOOKS_DF, count=2)
print_examples("Word2Vec Examples", word2vec_recommender, word2vec_example_titles, num_recommendations=5)

## BERT Embeddings
This model uses SentenceTransformer embeddings to capture deeper semantic similarity between books.

In [ ]:
bert_recommender = BertEmbeddingRecommender(
    BOOKS_DF,
    model_name="paraphrase-MiniLM-L3-v2",
    allow_download=False,
 )
bert_example_titles = choose_example_titles(BOOKS_DF, count=2)
print(f"BERT backend in use: {bert_recommender.backend}")
print_examples("BERT Embedding Examples", bert_recommender, bert_example_titles, num_recommendations=5)

## Evaluation
The same precision/recall procedure is applied to all three models so the results are directly comparable.

In [ ]:
models = {
    "Cosine Similarity": cosine_recommender,
    "Word2Vec": word2vec_recommender,
    "BERT": bert_recommender,
}

benchmark_rows = []
category_breakdowns = {}

for model_name, recommender in models.items():
    evaluator = PrecisionRecallEvaluator(recommender, BOOKS_DF)
    results = evaluator.evaluate_all_categories(num_queries=3, num_recommendations=5)
    category_rows = [
        {
            "Category": category,
            "Precision": result["avg_precision"],
            "Recall": result["avg_recall"],
            "F1": result["avg_f1"],
            "Queries": len(result["examples"]),
        }
        for category, result in results.items()
    ]
    category_df = pd.DataFrame(category_rows)
    category_breakdowns[model_name] = category_df

    avg_precision = float(category_df["Precision"].mean()) if not category_df.empty else 0.0
    avg_recall = float(category_df["Recall"].mean()) if not category_df.empty else 0.0
    avg_f1 = float(category_df["F1"].mean()) if not category_df.empty else 0.0


    benchmark_rows.append({
        "Model": model_name,
        "Avg Precision": avg_precision,
        "Avg Recall": avg_recall,
        "Avg F1": avg_f1,
        "Categories Evaluated": int(category_df.shape[0]),
    })

    print(f"\n{'=' * 100}")
    print(f"{model_name.upper()} EVALUATION")
    print(f"{'=' * 100}")
    print(f"Backend: {backend_label}")
    print(f"Average Precision: {avg_precision:.3f}")
    print(f"Average Recall:    {avg_recall:.3f}")
    print(f"Average F1 Score:   {avg_f1:.3f}")
    if not category_df.empty:
        print("\nSample category breakdown:")
        print(category_df.sort_values(["Precision", "Recall"], ascending=False).head(5).to_string(index=False))
    else:
        print("No categories were evaluable for this model.")

benchmark_df = pd.DataFrame(benchmark_rows)
print(f"\n{'=' * 100}")
print("MODEL COMPARISON SUMMARY")
print(f"{'=' * 100}")
print(benchmark_df.to_string(index=False))